# ZGLMM147 — Centro 4128 — Datalake

**Tabela:** `dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras`
**Transação SAP:** ZGLMM147 · **Domínio:** Follow-up de compras — divisoes de remessa do pedido
**Filtro do cenário:** `cod_centro = '4128'`
**Colunas:** 24 · **Clustering declarado:** `num_pedido_compra`, `cod_material`, `cod_centro`, `dt_criacao`

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** — cada uma consulta a tabela
diretamente com o filtro do centro embutido. Não há widget, view temporária nem ordem obrigatória.

## Alertas de partida _(dos comentários do schema + execução da exploratória)_

| Severidade | Alerta |
|---|---|
| 🔴 ALTA | 🔒 **LGPD — `nm_usuario_criacao`.** O schema instrui: *"nunca amostrar valor literal nem usar como literal em WHERE"*. Este notebook **mascara a coluna** e a analisa só por contagem. |
| 🔴 ALTA | **Nunca somar `qt_pedido` nem `vl_liquido` sem deduplicar** — são valores do **item** que se repetem em cada divisão de remessa. A seção 11 traz os dois totais lado a lado. |
| 🔴 ALTA | **`qt_a_fornecer` negativo é esperado** (9,6% das divisões) — over-delivery pela tolerância de excesso. Não é erro de carga. |
| 🟡 MEDIA | **Tipos de transferência (UB, ZUB, ZRE) não existem aqui.** O universo exclui pedidos com RESWK preenchido — aplicar o mesmo filtro na extração do SAP. |
| 🟡 MEDIA | **`ind_entrega_completa` é STRING** (`X` e vazio), não boolean. |

## Seções

| # | Conteúdo |
|---|---|
| 1 | Metadados |
| 2 | Volumetria e representatividade |
| 3 | Confirmação do filtro |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| **6** | **Preenchimento de todas as colunas** |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| 10 | Perfil numérico |
| **11** | **Totais para conciliação com o SAP** |
| 12 | Datas |
| 13 | Códigos e zeros à esquerda |
| **14** | **Chaves normalizadas para join** |
| **15** | **Checksum de linha** |
| 16 | Amostra |
| 17 | Distribuição interna |
| **18** | **Análises específicas da transação** |
| **19** | **EXTRAÇÃO COMPLETA** |
| 20 | Resumo do cenário |

> **Aviso:** contagem de linhas não é evidência de qualidade. Ver seções 4, 5 e 15.


## 1. Metadados

In [ ]:
DESCRIBE EXTENDED dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras;

In [ ]:
-- Formato, tamanho e particoes
DESCRIBE DETAIL dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras;

In [ ]:
-- Ultimas gravacoes
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras LIMIT 20;

## 2. Volumetria e representatividade

Quanto o centro 4128 representa do total.

In [ ]:
-- 2. VOLUMETRIA DO CENARIO
SELECT
  (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras)                            AS linhas_tabela_toda,
  (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128')                        AS linhas_do_cenario,
  ROUND(100.0 * (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128')
              / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras), 4)           AS pct_do_total,
  (SELECT COUNT(DISTINCT `cod_centro`) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras)              AS valores_distintos_na_tabela;

## 3. Confirmação do filtro

Confirma que `4128` existe e que não há variação de formato que faça o filtro perder linhas.

**Se retornar mais de uma linha com formatos diferentes, o filtro está incompleto.**

In [ ]:
-- 3. O FILTRO PEGOU TUDO?
SELECT CAST(`cod_centro` AS STRING)         AS valor_bruto,
       length(CAST(`cod_centro` AS STRING)) AS comprimento,
       COUNT(*)                        AS linhas
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
WHERE regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '') = '4128'
   OR trim(CAST(`cod_centro` AS STRING)) = '4128'
GROUP BY CAST(`cod_centro` AS STRING), length(CAST(`cod_centro` AS STRING))
ORDER BY linhas DESC;

## 4. Granularidade e chave real

`linhas ÷ chaves distintas`. Razão maior que 1,00 indica dimensão adicional.

> 🔴 **Na base completa, NENHUMA chave testada foi única.** Se o mesmo ocorrer aqui, a seção 18.1 amplia a busca.

In [ ]:
-- GRANULARIDADE NO CENARIO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'),
g AS (
  SELECT 'num_pedido_compra' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_pedido_compra` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128')
UNION ALL
  SELECT 'num_pedido_compra + num_item_pedido_compra' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_pedido_compra`, `num_item_pedido_compra` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128')
UNION ALL
  SELECT 'num_pedido_compra + num_item_pedido_compra + num_divisao_remessa' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_pedido_compra`, `num_item_pedido_compra`, `num_divisao_remessa` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128')
)
SELECT g.chave, t.total AS linhas, g.distintos,
       ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade

Analisando pela chave de clustering `num_pedido_compra + cod_material + cod_centro + dt_criacao`.

**Regra:** linhas idênticas = duplicata real. Linhas distintas = granularidade legítima.

> Na base completa **há duplicidade** por esta chave — é granularidade legítima.

In [ ]:
-- 5. CHAVES DUPLICADAS
SELECT `num_pedido_compra`, `cod_material`, `cod_centro`, `dt_criacao`, COUNT(*) AS qtd
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'
GROUP BY `num_pedido_compra`, `cod_material`, `cod_centro`, `dt_criacao`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 30;

In [ ]:
-- O QUE DIFERENCIA AS LINHAS DUPLICADAS
-- Colunas sob LGPD excluidas: nm_usuario_criacao
WITH cen AS (
  SELECT * FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'
),
dup AS (
  SELECT `num_pedido_compra`, `cod_material`, `cod_centro`, `dt_criacao` FROM cen GROUP BY `num_pedido_compra`, `cod_material`, `cod_centro`, `dt_criacao` HAVING COUNT(*) > 1
),
d AS (
  SELECT c.* FROM cen c JOIN dup ON c.`num_pedido_compra` <=> dup.`num_pedido_compra` AND c.`cod_material` <=> dup.`cod_material` AND c.`cod_centro` <=> dup.`cod_centro` AND c.`dt_criacao` <=> dup.`dt_criacao`
),
agg AS (
  SELECT `num_pedido_compra`, `cod_material`, `cod_centro`, `dt_criacao`,
         COUNT(DISTINCT `num_item_pedido_compra`) AS `num_item_pedido_compra`,
         COUNT(DISTINCT `num_divisao_remessa`) AS `num_divisao_remessa`,
         COUNT(DISTINCT `cod_grupo_compradores`) AS `cod_grupo_compradores`,
         COUNT(DISTINCT `nm_grupo_compradores`) AS `nm_grupo_compradores`,
         COUNT(DISTINCT `cod_grupo_mercadoria`) AS `cod_grupo_mercadoria`,
         COUNT(DISTINCT `qt_pedido`) AS `qt_pedido`,
         COUNT(DISTINCT `qt_a_fornecer`) AS `qt_a_fornecer`,
         COUNT(DISTINCT `cod_unidade_medida`) AS `cod_unidade_medida`,
         COUNT(DISTINCT `vl_liquido`) AS `vl_liquido`,
         COUNT(DISTINCT `qt_unidade_preco`) AS `qt_unidade_preco`,
         COUNT(DISTINCT `cod_moeda`) AS `cod_moeda`,
         COUNT(DISTINCT `ind_entrega_completa`) AS `ind_entrega_completa`,
         COUNT(DISTINCT `dt_remessa`) AS `dt_remessa`,
         COUNT(DISTINCT `cod_fornecedor`) AS `cod_fornecedor`,
         COUNT(DISTINCT `nm_fornecedor`) AS `nm_fornecedor`,
         COUNT(DISTINCT `cod_incoterms`) AS `cod_incoterms`,
         COUNT(DISTINCT `nm_incoterms_local`) AS `nm_incoterms_local`,
         COUNT(DISTINCT `cod_tipo_pedido_compra`) AS `cod_tipo_pedido_compra`,
         COUNT(DISTINCT `dateingest`) AS `dateingest`
  FROM d GROUP BY `num_pedido_compra`, `cod_material`, `cod_centro`, `dt_criacao`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(19,
    'num_item_pedido_compra', MAX(`num_item_pedido_compra`),
    'num_divisao_remessa', MAX(`num_divisao_remessa`),
    'cod_grupo_compradores', MAX(`cod_grupo_compradores`),
    'nm_grupo_compradores', MAX(`nm_grupo_compradores`),
    'cod_grupo_mercadoria', MAX(`cod_grupo_mercadoria`),
    'qt_pedido', MAX(`qt_pedido`),
    'qt_a_fornecer', MAX(`qt_a_fornecer`),
    'cod_unidade_medida', MAX(`cod_unidade_medida`),
    'vl_liquido', MAX(`vl_liquido`),
    'qt_unidade_preco', MAX(`qt_unidade_preco`),
    'cod_moeda', MAX(`cod_moeda`),
    'ind_entrega_completa', MAX(`ind_entrega_completa`),
    'dt_remessa', MAX(`dt_remessa`),
    'cod_fornecedor', MAX(`cod_fornecedor`),
    'nm_fornecedor', MAX(`nm_fornecedor`),
    'cod_incoterms', MAX(`cod_incoterms`),
    'nm_incoterms_local', MAX(`nm_incoterms_local`),
    'cod_tipo_pedido_compra', MAX(`cod_tipo_pedido_compra`),
    'dateingest', MAX(`dateingest`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Detecta coluna nunca carregada **neste centro**.

Uma coluna pode ter dado na tabela toda e estar vazia no centro 4128 — ou o contrário.

In [ ]:
-- PREENCHIMENTO DE TODAS AS COLUNAS NO CENARIO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'),
perf AS (
  SELECT stack(24,
    'num_pedido_compra', 'string', COUNT_IF(`num_pedido_compra` IS NULL), COUNT_IF(`num_pedido_compra` IS NOT NULL AND lower(trim(`num_pedido_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_pedido_compra`) RLIKE '^0+([.,]0+)?$'),
    'num_item_pedido_compra', 'string', COUNT_IF(`num_item_pedido_compra` IS NULL), COUNT_IF(`num_item_pedido_compra` IS NOT NULL AND lower(trim(`num_item_pedido_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_pedido_compra`) RLIKE '^0+([.,]0+)?$'),
    'num_divisao_remessa', 'string', COUNT_IF(`num_divisao_remessa` IS NULL), COUNT_IF(`num_divisao_remessa` IS NOT NULL AND lower(trim(`num_divisao_remessa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_divisao_remessa`) RLIKE '^0+([.,]0+)?$'),
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'cod_grupo_compradores', 'string', COUNT_IF(`cod_grupo_compradores` IS NULL), COUNT_IF(`cod_grupo_compradores` IS NOT NULL AND lower(trim(`cod_grupo_compradores`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_grupo_compradores`) RLIKE '^0+([.,]0+)?$'),
    'nm_grupo_compradores', 'string', COUNT_IF(`nm_grupo_compradores` IS NULL), COUNT_IF(`nm_grupo_compradores` IS NOT NULL AND lower(trim(`nm_grupo_compradores`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_grupo_compradores`) RLIKE '^0+([.,]0+)?$'),
    'nm_usuario_criacao', 'string', COUNT_IF(`nm_usuario_criacao` IS NULL), COUNT_IF(`nm_usuario_criacao` IS NOT NULL AND lower(trim(`nm_usuario_criacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_usuario_criacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_grupo_mercadoria', 'string', COUNT_IF(`cod_grupo_mercadoria` IS NULL), COUNT_IF(`cod_grupo_mercadoria` IS NOT NULL AND lower(trim(`cod_grupo_mercadoria`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_grupo_mercadoria`) RLIKE '^0+([.,]0+)?$'),
    'qt_pedido', 'decimal(13,3)', COUNT_IF(`qt_pedido` IS NULL), 0L, COUNT_IF(`qt_pedido` = 0),
    'qt_a_fornecer', 'decimal(14,3)', COUNT_IF(`qt_a_fornecer` IS NULL), 0L, COUNT_IF(`qt_a_fornecer` = 0),
    'cod_unidade_medida', 'string', COUNT_IF(`cod_unidade_medida` IS NULL), COUNT_IF(`cod_unidade_medida` IS NOT NULL AND lower(trim(`cod_unidade_medida`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_unidade_medida`) RLIKE '^0+([.,]0+)?$'),
    'vl_liquido', 'double', COUNT_IF(`vl_liquido` IS NULL), 0L, COUNT_IF(`vl_liquido` = 0),
    'qt_unidade_preco', 'decimal(5,0)', COUNT_IF(`qt_unidade_preco` IS NULL), 0L, COUNT_IF(`qt_unidade_preco` = 0),
    'cod_moeda', 'string', COUNT_IF(`cod_moeda` IS NULL), COUNT_IF(`cod_moeda` IS NOT NULL AND lower(trim(`cod_moeda`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_moeda`) RLIKE '^0+([.,]0+)?$'),
    'ind_entrega_completa', 'string', COUNT_IF(`ind_entrega_completa` IS NULL), COUNT_IF(`ind_entrega_completa` IS NOT NULL AND lower(trim(`ind_entrega_completa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_entrega_completa`) RLIKE '^0+([.,]0+)?$'),
    'dt_remessa', 'date', COUNT_IF(`dt_remessa` IS NULL), 0L, 0L,
    'dt_criacao', 'date', COUNT_IF(`dt_criacao` IS NULL), 0L, 0L,
    'cod_fornecedor', 'string', COUNT_IF(`cod_fornecedor` IS NULL), COUNT_IF(`cod_fornecedor` IS NOT NULL AND lower(trim(`cod_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'nm_fornecedor', 'string', COUNT_IF(`nm_fornecedor` IS NULL), COUNT_IF(`nm_fornecedor` IS NOT NULL AND lower(trim(`nm_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'cod_incoterms', 'string', COUNT_IF(`cod_incoterms` IS NULL), COUNT_IF(`cod_incoterms` IS NOT NULL AND lower(trim(`cod_incoterms`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_incoterms`) RLIKE '^0+([.,]0+)?$'),
    'nm_incoterms_local', 'string', COUNT_IF(`nm_incoterms_local` IS NULL), COUNT_IF(`nm_incoterms_local` IS NOT NULL AND lower(trim(`nm_incoterms_local`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_incoterms_local`) RLIKE '^0+([.,]0+)?$'),
    'cod_tipo_pedido_compra', 'string', COUNT_IF(`cod_tipo_pedido_compra` IS NULL), COUNT_IF(`cod_tipo_pedido_compra` IS NOT NULL AND lower(trim(`cod_tipo_pedido_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_tipo_pedido_compra`) RLIKE '^0+([.,]0+)?$'),
    'dateingest', 'date', COUNT_IF(`dateingest` IS NULL), 0L, 0L
  ) AS (coluna, tipo, nulos, vazios, zeros_ou_false)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'
)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros_ou_false,
       t.total - p.nulos - p.vazios - p.zeros_ou_false                               AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros_ou_false) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total                                               THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros_ou_false <= 0             THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros_ou_false) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END                                                      AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade no cenário

In [ ]:
-- CARDINALIDADE NO CENARIO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'),
card AS (
  SELECT stack(24,
    'num_pedido_compra', 'string', approx_count_distinct(`num_pedido_compra`),
    'num_item_pedido_compra', 'string', approx_count_distinct(`num_item_pedido_compra`),
    'num_divisao_remessa', 'string', approx_count_distinct(`num_divisao_remessa`),
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'cod_grupo_compradores', 'string', approx_count_distinct(`cod_grupo_compradores`),
    'nm_grupo_compradores', 'string', approx_count_distinct(`nm_grupo_compradores`),
    'nm_usuario_criacao', 'string', approx_count_distinct(`nm_usuario_criacao`),
    'cod_grupo_mercadoria', 'string', approx_count_distinct(`cod_grupo_mercadoria`),
    'qt_pedido', 'decimal(13,3)', approx_count_distinct(`qt_pedido`),
    'qt_a_fornecer', 'decimal(14,3)', approx_count_distinct(`qt_a_fornecer`),
    'cod_unidade_medida', 'string', approx_count_distinct(`cod_unidade_medida`),
    'vl_liquido', 'double', approx_count_distinct(`vl_liquido`),
    'qt_unidade_preco', 'decimal(5,0)', approx_count_distinct(`qt_unidade_preco`),
    'cod_moeda', 'string', approx_count_distinct(`cod_moeda`),
    'ind_entrega_completa', 'string', approx_count_distinct(`ind_entrega_completa`),
    'dt_remessa', 'date', approx_count_distinct(`dt_remessa`),
    'dt_criacao', 'date', approx_count_distinct(`dt_criacao`),
    'cod_fornecedor', 'string', approx_count_distinct(`cod_fornecedor`),
    'nm_fornecedor', 'string', approx_count_distinct(`nm_fornecedor`),
    'cod_incoterms', 'string', approx_count_distinct(`cod_incoterms`),
    'nm_incoterms_local', 'string', approx_count_distinct(`nm_incoterms_local`),
    'cod_tipo_pedido_compra', 'string', approx_count_distinct(`cod_tipo_pedido_compra`),
    'dateingest', 'date', approx_count_distinct(`dateingest`)
  ) AS (coluna, tipo, distintos)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1             THEN '1. CONSTANTE'
            WHEN c.distintos <= 3             THEN '2. cardinalidade muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador'
            ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada uma.

> 🔒 Colunas sob LGPD excluídas: `nm_usuario_criacao`

In [ ]:
-- DOMINIO DAS CATEGORICAS NO CENARIO
(SELECT 'cod_grupo_compradores' AS coluna, CAST(`cod_grupo_compradores` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128' GROUP BY `cod_grupo_compradores` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_grupo_mercadoria' AS coluna, CAST(`cod_grupo_mercadoria` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128' GROUP BY `cod_grupo_mercadoria` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_unidade_medida' AS coluna, CAST(`cod_unidade_medida` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128' GROUP BY `cod_unidade_medida` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_moeda' AS coluna, CAST(`cod_moeda` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128' GROUP BY `cod_moeda` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_entrega_completa' AS coluna, CAST(`ind_entrega_completa` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128' GROUP BY `ind_entrega_completa` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_incoterms' AS coluna, CAST(`cod_incoterms` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128' GROUP BY `cod_incoterms` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_tipo_pedido_compra' AS coluna, CAST(`cod_tipo_pedido_compra` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128' GROUP BY `cod_tipo_pedido_compra` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 10. Perfil dos campos numéricos

**1 colunas `double`** entre as analisadas — tolerância de 0,005.

In [ ]:
-- PERFIL NUMERICO NO CENARIO
SELECT * FROM (
  SELECT stack(4,
    'qt_pedido', 'decimal(13,3)', COUNT(`qt_pedido`), CAST(MIN(`qt_pedido`) AS DOUBLE), CAST(MAX(`qt_pedido`) AS DOUBLE), CAST(AVG(`qt_pedido`) AS DOUBLE), CAST(percentile_approx(`qt_pedido`, 0.5) AS DOUBLE), COUNT_IF(`qt_pedido` < 0), COUNT_IF(`qt_pedido` = 0),
    'qt_a_fornecer', 'decimal(14,3)', COUNT(`qt_a_fornecer`), CAST(MIN(`qt_a_fornecer`) AS DOUBLE), CAST(MAX(`qt_a_fornecer`) AS DOUBLE), CAST(AVG(`qt_a_fornecer`) AS DOUBLE), CAST(percentile_approx(`qt_a_fornecer`, 0.5) AS DOUBLE), COUNT_IF(`qt_a_fornecer` < 0), COUNT_IF(`qt_a_fornecer` = 0),
    'vl_liquido', 'double', COUNT(`vl_liquido`), CAST(MIN(`vl_liquido`) AS DOUBLE), CAST(MAX(`vl_liquido`) AS DOUBLE), CAST(AVG(`vl_liquido`) AS DOUBLE), CAST(percentile_approx(`vl_liquido`, 0.5) AS DOUBLE), COUNT_IF(`vl_liquido` < 0), COUNT_IF(`vl_liquido` = 0),
    'qt_unidade_preco', 'decimal(5,0)', COUNT(`qt_unidade_preco`), CAST(MIN(`qt_unidade_preco`) AS DOUBLE), CAST(MAX(`qt_unidade_preco`) AS DOUBLE), CAST(AVG(`qt_unidade_preco`) AS DOUBLE), CAST(percentile_approx(`qt_unidade_preco`, 0.5) AS DOUBLE), COUNT_IF(`qt_unidade_preco` < 0), COUNT_IF(`qt_unidade_preco` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'
)
ORDER BY coluna;

## 11. Totais para conciliação com o SAP

**Use esta tabela para bater os totais contra o extrato do SAP.**

Divergência de total detecta registro faltando ou duplicado — cobre o ponto cego
da contagem de linhas.

> ⚠️ **`qt_pedido` e `vl_liquido` estão INFLADOS nesta tabela** — repetem-se em cada divisão de remessa. Use a seção 18.3, que traz a soma deduplicada.

In [ ]:
-- TOTAIS PARA CONCILIACAO COM O SAP
SELECT coluna, total_numerico, total_arredondado, linhas_preenchidas
FROM (
  SELECT stack(4,
    'qt_pedido', CAST(SUM(`qt_pedido`) AS DOUBLE), CAST(ROUND(SUM(`qt_pedido`), 2) AS STRING), COUNT(`qt_pedido`),
    'qt_a_fornecer', CAST(SUM(`qt_a_fornecer`) AS DOUBLE), CAST(ROUND(SUM(`qt_a_fornecer`), 2) AS STRING), COUNT(`qt_a_fornecer`),
    'vl_liquido', CAST(SUM(`vl_liquido`) AS DOUBLE), CAST(ROUND(SUM(`vl_liquido`), 2) AS STRING), COUNT(`vl_liquido`),
    'qt_unidade_preco', CAST(SUM(`qt_unidade_preco`) AS DOUBLE), CAST(ROUND(SUM(`qt_unidade_preco`), 2) AS STRING), COUNT(`qt_unidade_preco`)
  ) AS (coluna, total_numerico, total_arredondado, linhas_preenchidas)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'
)
ORDER BY coluna;

## 12. Datas

Todas em tipo nativo `date`. A coluna `sentinela_9999` conta ocorrências de `9999-12-31`.

In [ ]:
-- DATAS EM TIPO NATIVO
SELECT coluna, nulos, minimo, maximo, distintas, futuras, sentinela_9999
FROM (
  SELECT stack(3,
    'dt_remessa', COUNT_IF(`dt_remessa` IS NULL), CAST(MIN(`dt_remessa`) AS STRING), CAST(MAX(`dt_remessa`) AS STRING), COUNT(DISTINCT `dt_remessa`), COUNT_IF(`dt_remessa` > current_date()), COUNT_IF(`dt_remessa` = DATE'9999-12-31'),
    'dt_criacao', COUNT_IF(`dt_criacao` IS NULL), CAST(MIN(`dt_criacao`) AS STRING), CAST(MAX(`dt_criacao`) AS STRING), COUNT(DISTINCT `dt_criacao`), COUNT_IF(`dt_criacao` > current_date()), COUNT_IF(`dt_criacao` = DATE'9999-12-31'),
    'dateingest', COUNT_IF(`dateingest` IS NULL), CAST(MIN(`dateingest`) AS STRING), CAST(MAX(`dateingest`) AS STRING), COUNT(DISTINCT `dateingest`), COUNT_IF(`dateingest` > current_date()), COUNT_IF(`dateingest` = DATE'9999-12-31')
  ) AS (coluna, nulos, minimo, maximo, distintas, futuras, sentinela_9999)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'
)
ORDER BY coluna;

## 13. Códigos — zeros à esquerda e formato

**Armadilha:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá 0% de match.

In [ ]:
-- CODIGOS: ZEROS A ESQUERDA E FORMATO
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes,
       CONCAT_WS(' | ',
         CASE WHEN tipo LIKE 'big%' OR tipo LIKE '%int%'
              THEN 'TIPO NUMERICO - zeros ja perdidos' END,
         CASE WHEN com_zeros_esq > 0 THEN 'normalizar antes do join' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END
       ) AS alertas
FROM (
  SELECT stack(6,
    'num_pedido_compra', 'string', COUNT_IF(CAST(`num_pedido_compra` AS STRING) IS NULL OR trim(CAST(`num_pedido_compra` AS STRING)) = ''), MIN(length(trim(CAST(`num_pedido_compra` AS STRING)))), MAX(length(trim(CAST(`num_pedido_compra` AS STRING)))), COUNT_IF(trim(CAST(`num_pedido_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_pedido_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_pedido_compra` AS STRING)), '^0+', '')),
    'num_item_pedido_compra', 'string', COUNT_IF(CAST(`num_item_pedido_compra` AS STRING) IS NULL OR trim(CAST(`num_item_pedido_compra` AS STRING)) = ''), MIN(length(trim(CAST(`num_item_pedido_compra` AS STRING)))), MAX(length(trim(CAST(`num_item_pedido_compra` AS STRING)))), COUNT_IF(trim(CAST(`num_item_pedido_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_item_pedido_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_item_pedido_compra` AS STRING)), '^0+', '')),
    'num_divisao_remessa', 'string', COUNT_IF(CAST(`num_divisao_remessa` AS STRING) IS NULL OR trim(CAST(`num_divisao_remessa` AS STRING)) = ''), MIN(length(trim(CAST(`num_divisao_remessa` AS STRING)))), MAX(length(trim(CAST(`num_divisao_remessa` AS STRING)))), COUNT_IF(trim(CAST(`num_divisao_remessa` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_divisao_remessa` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_divisao_remessa` AS STRING)), '^0+', '')),
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'cod_centro', 'string', COUNT_IF(CAST(`cod_centro` AS STRING) IS NULL OR trim(CAST(`cod_centro` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro` AS STRING)))), MAX(length(trim(CAST(`cod_centro` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_centro` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '')),
    'cod_fornecedor', 'string', COUNT_IF(CAST(`cod_fornecedor` AS STRING) IS NULL OR trim(CAST(`cod_fornecedor` AS STRING)) = ''), MIN(length(trim(CAST(`cod_fornecedor` AS STRING)))), MAX(length(trim(CAST(`cod_fornecedor` AS STRING)))), COUNT_IF(trim(CAST(`cod_fornecedor` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_fornecedor` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_fornecedor` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
        distintos_bruto, distintos_sem_zeros)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'
)
ORDER BY coluna;

## 14. Chaves normalizadas para join com o SAP

Chave `num_pedido_compra + num_item_pedido_compra + num_divisao_remessa` já **sem zeros à esquerda**, pronta para cruzar
com o extrato do SAP via PROCV/ÍNDICE.

In [ ]:
-- 14. CHAVES NORMALIZADAS
SELECT DISTINCT
       regexp_replace(trim(CAST(`num_pedido_compra` AS STRING)), '^0+', '') AS `num_pedido_compra_norm`,
       regexp_replace(trim(CAST(`num_item_pedido_compra` AS STRING)), '^0+', '') AS `num_item_pedido_compra_norm`,
       regexp_replace(trim(CAST(`num_divisao_remessa` AS STRING)), '^0+', '') AS `num_divisao_remessa_norm`
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'
ORDER BY 1, 2, 3;

## 15. Checksum de linha

Se `linhas` > `linhas_unicas`, existem registros 100% idênticos — duplicata real.

In [ ]:
-- CHECKSUM DE LINHA
SELECT COUNT(*)                                            AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_item_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_divisao_remessa` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_grupo_compradores` AS STRING), ''), COALESCE(CAST(`nm_grupo_compradores` AS STRING), ''), COALESCE(CAST(`nm_usuario_criacao` AS STRING), ''), COALESCE(CAST(`cod_grupo_mercadoria` AS STRING), ''), COALESCE(CAST(`qt_pedido` AS STRING), ''), COALESCE(CAST(`qt_a_fornecer` AS STRING), ''), COALESCE(CAST(`cod_unidade_medida` AS STRING), ''), COALESCE(CAST(`vl_liquido` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`ind_entrega_completa` AS STRING), ''), COALESCE(CAST(`dt_remessa` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_incoterms` AS STRING), ''), COALESCE(CAST(`nm_incoterms_local` AS STRING), ''), COALESCE(CAST(`cod_tipo_pedido_compra` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''))))            AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_item_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_divisao_remessa` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_grupo_compradores` AS STRING), ''), COALESCE(CAST(`nm_grupo_compradores` AS STRING), ''), COALESCE(CAST(`nm_usuario_criacao` AS STRING), ''), COALESCE(CAST(`cod_grupo_mercadoria` AS STRING), ''), COALESCE(CAST(`qt_pedido` AS STRING), ''), COALESCE(CAST(`qt_a_fornecer` AS STRING), ''), COALESCE(CAST(`cod_unidade_medida` AS STRING), ''), COALESCE(CAST(`vl_liquido` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`ind_entrega_completa` AS STRING), ''), COALESCE(CAST(`dt_remessa` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_incoterms` AS STRING), ''), COALESCE(CAST(`nm_incoterms_local` AS STRING), ''), COALESCE(CAST(`cod_tipo_pedido_compra` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), '')))) AS linhas_100pct_identicas,
       CASE WHEN COUNT(*) = COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_item_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_divisao_remessa` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_grupo_compradores` AS STRING), ''), COALESCE(CAST(`nm_grupo_compradores` AS STRING), ''), COALESCE(CAST(`nm_usuario_criacao` AS STRING), ''), COALESCE(CAST(`cod_grupo_mercadoria` AS STRING), ''), COALESCE(CAST(`qt_pedido` AS STRING), ''), COALESCE(CAST(`qt_a_fornecer` AS STRING), ''), COALESCE(CAST(`cod_unidade_medida` AS STRING), ''), COALESCE(CAST(`vl_liquido` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`ind_entrega_completa` AS STRING), ''), COALESCE(CAST(`dt_remessa` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_incoterms` AS STRING), ''), COALESCE(CAST(`nm_incoterms_local` AS STRING), ''), COALESCE(CAST(`cod_tipo_pedido_compra` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''))))
            THEN 'OK - nenhuma linha totalmente identica'
            ELSE 'ATENCAO - existem linhas identicas em todos os campos' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128';

## 16. Amostra de linhas completas

> 🔒 Colunas sob LGPD mascaradas por hash.

In [ ]:
-- AMOSTRA (colunas sob LGPD mascaradas)
SELECT `num_pedido_compra`,
       `num_item_pedido_compra`,
       `num_divisao_remessa`,
       `cod_material`,
       `cod_centro`,
       `cod_grupo_compradores`,
       `nm_grupo_compradores`,
       substring(md5(`nm_usuario_criacao`), 1, 8) AS `nm_usuario_criacao_hash`  -- LGPD,
       `cod_grupo_mercadoria`,
       `qt_pedido`,
       `qt_a_fornecer`,
       `cod_unidade_medida`,
       `vl_liquido`,
       `qt_unidade_preco`,
       `cod_moeda`,
       `ind_entrega_completa`,
       `dt_remessa`,
       `dt_criacao`,
       `cod_fornecedor`,
       `nm_fornecedor`,
       `cod_incoterms`,
       `nm_incoterms_local`,
       `cod_tipo_pedido_compra`,
       `dateingest`
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'
ORDER BY `num_pedido_compra`, `num_item_pedido_compra`, `num_divisao_remessa`
LIMIT 20;

## 17. Distribuição interna do centro 4128

Útil para conferir se o extrato do SAP tem a mesma composição.

In [ ]:
-- 17. DISTRIBUICAO POR cod_grupo_compradores
SELECT COALESCE(NULLIF(trim(CAST(`cod_grupo_compradores` AS STRING)), ''), '(vazio)') AS `cod_grupo_compradores`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_grupo_compradores` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [ ]:
-- 17. DISTRIBUICAO POR cod_tipo_pedido_compra
SELECT COALESCE(NULLIF(trim(CAST(`cod_tipo_pedido_compra` AS STRING)), ''), '(vazio)') AS `cod_tipo_pedido_compra`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_tipo_pedido_compra` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [ ]:
-- 17. DISTRIBUICAO POR cod_grupo_mercadoria
SELECT COALESCE(NULLIF(trim(CAST(`cod_grupo_mercadoria` AS STRING)), ''), '(vazio)') AS `cod_grupo_mercadoria`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_grupo_mercadoria` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [ ]:
-- 17. DISTRIBUICAO POR cod_moeda
SELECT COALESCE(NULLIF(trim(CAST(`cod_moeda` AS STRING)), ''), '(vazio)') AS `cod_moeda`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_moeda` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

### 17.1 Freshness

In [ ]:
-- 17.1 FRESHNESS
SELECT `dateingest` AS data_carga, COUNT(*) AS linhas
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'
GROUP BY `dateingest`
ORDER BY data_carga DESC
LIMIT 30;

## 18. Análises específicas — ZGLMM147

### 18.1 Divisões de remessa por item

O grão da tabela é a **divisão de remessa**. Item sem programação tem
`num_divisao_remessa` nulo — e nesse caso `dt_remessa` e `qt_a_fornecer` também são nulos.

In [ ]:
-- 18.1 DIVISOES POR ITEM
SELECT divisoes_no_item, COUNT(*) AS itens
FROM (SELECT num_pedido_compra, num_item_pedido_compra,
             COUNT(*) AS divisoes_no_item
        FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'
       GROUP BY num_pedido_compra, num_item_pedido_compra)
GROUP BY divisoes_no_item
ORDER BY divisoes_no_item
LIMIT 40;

### 18.2 `qt_a_fornecer` — saldo negativo é esperado

Domínio medido em 2026-09-09 sobre 7.513.595 divisões: zero 68,4%, positivo 22,0%,
**negativo 9,6%** (over-delivery). Não é erro de carga.

In [ ]:
-- 18.2 DOMINIO DE qt_a_fornecer
SELECT COUNT(*)                                           AS divisoes,
       COUNT_IF(qt_a_fornecer = 0)                        AS saldo_zero,
       ROUND(100.0 * COUNT_IF(qt_a_fornecer = 0) / COUNT(*), 2)  AS pct_zero,
       COUNT_IF(qt_a_fornecer > 0)                        AS saldo_positivo,
       ROUND(100.0 * COUNT_IF(qt_a_fornecer > 0) / COUNT(*), 2)  AS pct_positivo,
       COUNT_IF(qt_a_fornecer < 0)                        AS saldo_negativo,
       ROUND(100.0 * COUNT_IF(qt_a_fornecer < 0) / COUNT(*), 2)  AS pct_negativo,
       COUNT_IF(qt_a_fornecer IS NULL)                    AS sem_divisao,
       ROUND(MIN(qt_a_fornecer), 3)                       AS menor_saldo
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128';

### 18.3 ⚠️ Totais COM e SEM deduplicação

**Use sempre a coluna deduplicada para bater contra o SAP.**
A diferença entre as duas é o erro que uma soma direta produziria.

In [ ]:
-- 18.3 TOTAIS: SOMA INGENUA vs DEDUPLICADA
WITH dedup AS (
  SELECT num_pedido_compra, num_item_pedido_compra,
         MAX(qt_pedido)  AS qt_pedido,
         MAX(vl_liquido) AS vl_liquido
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'
  GROUP BY num_pedido_compra, num_item_pedido_compra
)
SELECT 'qt_pedido' AS coluna,
       (SELECT ROUND(SUM(qt_pedido), 3) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128')   AS soma_ingenua_INFLADA,
       (SELECT ROUND(SUM(qt_pedido), 3) FROM dedup)     AS soma_correta_dedup,
       (SELECT ROUND(SUM(qt_pedido), 3) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128')
     - (SELECT ROUND(SUM(qt_pedido), 3) FROM dedup)     AS diferenca
UNION ALL
SELECT 'vl_liquido',
       (SELECT ROUND(SUM(vl_liquido), 2) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'),
       (SELECT ROUND(SUM(vl_liquido), 2) FROM dedup),
       (SELECT ROUND(SUM(vl_liquido), 2) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128')
     - (SELECT ROUND(SUM(vl_liquido), 2) FROM dedup);

### 18.4 `ind_entrega_completa` — STRING, não boolean

Domínio declarado: `X` e vazio.

In [ ]:
-- 18.4 DOMINIO DE ind_entrega_completa
SELECT COALESCE(NULLIF(trim(ind_entrega_completa), ''), '(vazio)') AS ind_entrega_completa,
       COUNT(*)                                     AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'), 2) AS pct,
       COUNT_IF(qt_a_fornecer > 0)                  AS com_saldo_a_fornecer,
       CASE WHEN trim(COALESCE(ind_entrega_completa,'')) NOT IN ('X', '')
            THEN 'FORA DO DOMINIO' ELSE 'dominio esperado' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'
GROUP BY COALESCE(NULLIF(trim(ind_entrega_completa), ''), '(vazio)'),
         trim(COALESCE(ind_entrega_completa,''))
ORDER BY linhas DESC;

### 18.5 Escopo: transferências não existem aqui

O universo exclui pedidos com centro de saída de estoque (RESWK) preenchido.
**Aplique o mesmo filtro na extração do SAP.**

In [ ]:
-- 18.5 TIPOS DE PEDIDO PRESENTES
SELECT COALESCE(NULLIF(trim(cod_tipo_pedido_compra), ''), '(vazio)') AS cod_tipo_pedido_compra,
       COUNT(*)                                  AS linhas,
       COUNT(DISTINCT num_pedido_compra)         AS pedidos,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'), 2) AS pct,
       CASE WHEN trim(COALESCE(cod_tipo_pedido_compra,'')) IN ('UB','ZUB','ZRE')
            THEN 'TRANSFERENCIA - nao deveria existir aqui' ELSE 'ok' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'
GROUP BY COALESCE(NULLIF(trim(cod_tipo_pedido_compra), ''), '(vazio)'),
         trim(COALESCE(cod_tipo_pedido_compra,''))
ORDER BY linhas DESC
LIMIT 30;

### 18.6 🔒 Usuário de criação — somente agregados (LGPD)

Nenhum valor literal é exibido.

In [ ]:
-- 18.6 USUARIO DE CRIACAO (LGPD - somente agregados)
SELECT COUNT(*)                                                  AS linhas,
       COUNT(DISTINCT nm_usuario_criacao)                        AS usuarios_distintos,
       COUNT_IF(trim(COALESCE(nm_usuario_criacao,'')) = '')      AS sem_usuario,
       ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT nm_usuario_criacao), 0), 1) AS media_linhas_por_usuario
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128';

### 18.7 Sentinela 9999-12-31 e cobertura do fornecedor

In [ ]:
-- 18.7 SENTINELA E COBERTURA
SELECT COUNT(*)                                        AS divisoes,
       COUNT_IF(dt_remessa = DATE'9999-12-31')         AS sem_prazo_definido,
       COUNT_IF(dt_remessa IS NULL)                    AS dt_remessa_nula,
       COUNT_IF(dt_remessa < dt_criacao)               AS remessa_antes_da_criacao,
       COUNT_IF(trim(COALESCE(nm_fornecedor,'')) <> '') AS com_nome_fornecedor,
       ROUND(100.0 * COUNT_IF(trim(COALESCE(nm_fornecedor,'')) <> '') / COUNT(*), 2) AS pct_cobertura
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128';

## 19. EXTRAÇÃO COMPLETA — centro 4128

**Esta é a célula que você baixa para comparar com o SAP.**

Após executar, use **Download → CSV** no resultado.

> **Limites do Databricks:** a tela mostra até 10.000 linhas, mas o download em CSV
> vai além disso. Se o volume for grande, use a célula 19.1.

In [ ]:
-- 19. EXTRACAO COMPLETA DO CENARIO
SELECT *
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
WHERE `cod_centro` = '4128'
ORDER BY `num_pedido_compra`, `num_item_pedido_compra`, `num_divisao_remessa`;

### 19.1 Alternativa para volume grande _(opcional)_

In [ ]:
-- 19.1 GRAVAR EXTRACAO EM TABELA (opcional)
-- CREATE OR REPLACE TABLE dev_procurement.corp_curated.extracao_zglmm147_4128 AS
-- SELECT * FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128';
--
-- SELECT COUNT(*) FROM dev_procurement.corp_curated.extracao_zglmm147_4128;
SELECT 'Descomente as linhas acima se precisar gravar a extracao em tabela' AS instrucao;

## 20. Resumo do cenário

**Copie esta saída** e envie ao agente junto com o notebook.

In [ ]:
-- 20. RESUMO DO CENARIO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128'),
g AS (
  SELECT 'num_pedido_compra' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_pedido_compra` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128')
UNION ALL
  SELECT 'num_pedido_compra + num_item_pedido_compra' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_pedido_compra`, `num_item_pedido_compra` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128')
UNION ALL
  SELECT 'num_pedido_compra + num_item_pedido_compra + num_divisao_remessa' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_pedido_compra`, `num_item_pedido_compra`, `num_divisao_remessa` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras WHERE `cod_centro` = '4128')
)
SELECT 'CENARIO' AS bloco, 'transacao' AS item, 'ZGLMM147' AS valor
UNION ALL SELECT 'CENARIO', 'tabela', 'tbl_ds_pro_zglmm147_followup_compras'
UNION ALL SELECT 'CENARIO', 'coluna de recorte', 'cod_centro'
UNION ALL SELECT 'CENARIO', 'filtro', 'cod_centro = 4128'
UNION ALL SELECT 'CENARIO', 'linhas no cenario', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'CENARIO', 'colunas', '24'
UNION ALL
SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ',
              CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL
SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total),
                'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Baixar a **seção 19** em CSV — é a base do centro 4128 no Datalake.
2. Extrair a ZGLMM147 no SAP com o filtro equivalente ao centro 4128.
3. Anotar a data e hora das duas extrações.
4. Enviar ao agente de validação: este notebook executado + os arquivos do SAP.

### Antes de comparar

- [ ] Chave real confirmada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP (seção 6)
- [ ] Zeros à esquerda normalizados nos dois lados (seção 13)
- [ ] Tolerância de 0,005 aplicada em campos `double` (seção 10)
- [ ] Totais numéricos conferidos (seção 11)
- [ ] 🔒 Colunas sob LGPD tratadas só por agregado
- [ ] ⚠️ `qt_pedido` e `vl_liquido` deduplicados antes de somar (seção 18.3)
- [ ] Filtro de transferência (RESWK) aplicado também no SAP
